In [ ]:
import os
import re
import time
import pandas as pd
from openai import OpenAI, APITimeoutError, APIConnectionError
import dotenv
import json

dotenv.load_dotenv()

TYPHOON_API_KEY = os.getenv("TYPHOON_API_KEY")

client = OpenAI(api_key=TYPHOON_API_KEY, base_url="https://api.opentyphoon.ai/v1", timeout=60.0)

FILTER_PROMPT = """คุณเป็นผู้เชี่ยวชาญในการจำแนกประเภทของข้อมูลจากโปรแกรมอสังหาริมทรัพย์ในภาษาไทย

หน้าที่ของคุณคือ:
ตรวจสอบว่าเนื้อหาของ Post เกี่ยวข้องกับอสังหาริมทรัพย์หรือไม่

ถ้าเกี่ยวข้องกับอสังหาริมทรัพย์ (บ้าน, คอนโด, ที่ดิน, ทาวน์โฮม, ทาวน์เฮ้าส์, อพาร์ตเมนต์, วิลล่า, อาคารพาณิชย์, ออฟฟิศ, โฮมออฟฟิศ, โกดัง, โรงงาน) ให้ตอบ:
{"is_real_estate": true}

ถ้าไม่เกี่ยวข้องกับอสังหาริมทรัพย์ให้ตอบ:
{"is_real_estate": false}

ตอบเป็น JSON เพียงบรรทัดเดียวเท่านั้น ห้ามมีข้อความอื่น"""

SYSTEM_PROMPT = """คุณเป็นผู้เชี่ยวชาญวิเคราะห์โครงสร้างและดึงข้อมูลจากประกาศอสังหาริมทรัพย์ภาษาไทย เพื่อสร้างตารางสรุป 5 คอลัมน์หลัก คือ ประเภท, สถานะ, ราคา, ชื่อโครงการ/ทำเล, Link

หน้าที่หลักของคุณคือ:
1. วิเคราะห์ข้อความในประกาศอย่างละเอียด
2. ดึงรายละเอียดโครงสร้างให้ครบถ้วนที่สุด
3. แยกประเภท อสังหาริมทรัพย์ สถานะ (เช่า/ขาย) และทำเลตำแหน่งโดยละเอียด

การดึงข้อมูลโครงสร้างใน extracted:
- property_type:
  - ระบุประเภทอสังหาให้ชัดเจน เช่น "บ้านเดี่ยว", "บ้านแฝด", "ทาวน์เฮ้าส์", "คอนโด", "อาคารพาณิชย์", "ที่ดิน", "วิลล่า", "อพาร์ตเมนต์", "โฮมออฟฟิศ", หรือ "อื่นๆ" ที่ใกล้เคียงที่สุด
  - ถ้าไม่สามารถระบุประเภทได้ให้ใส่ "ไม่ระบุ"

- rental_sale_status:
  - ระบุว่าเป็นการเช่า (เช่า/rental/rent) หรือการขาย (ขาย/ขายอสังหาริมทรัพย์)
  - ค้นหาคำหลักเช่น "ขาย", "เช่า", "ให้เช่า", "ซื้อ", "rental", "sale", "for rent", "for sale"
  - หากมีคำเช่า ในประกาศให้ใส่ "เช่า" มิฉะนั้นให้ใส่ "ขาย"
  - ถ้าไม่สามารถระบุได้ให้ใส่ "ไม่ระบุ"

- price_text และ price_value_thb:
  - อ่านรูปแบบราคาไทยทุกแบบ เช่น "ราคา 17,900,000 บาท", "ราคาเพียง 1.6 ล้านบาท", "ขายเพียง 2,200,000 บาท", "2.5ล.", "เช่า 5,000 บาท/เดือน"
  - เก็บข้อความราคาเต็มดั้งเดิมไว้ใน price_text
  - แปลงเป็นจำนวนเงินหน่วยบาทใน price_value_thb (float) ถ้าไม่มีราคาให้ใส่ null

- project_name:
  - ดึงชื่อโครงการ/หมู่บ้าน/คอนโด/อาคารที่เป็นชื่อเฉพาะ (เช่น "IDEO MOBI Rama 9", "Laviq Sukhumvit57", "The Next Condominium", "Dusit Residence")
  - ถ้าไม่มีชื่อโครงการเฉพาะให้ใส่ "ไม่ระบุ"

- location_text:
  - ดึงข้อมูลตำแหน่งอย่างละเอียดและครอบคลุมจากเนื้อหา Post เช่น:
    * ที่อยู่ละเอียด (เลขที่, ซอย, ถนน, แขวง/ตำบล, เขต/อำเภอ, จังหวัด)
    * สถานีรถไฟฟ้า/สถานีรถไฟ (เช่น "ติดรถไฟฟ้า BTS", "ใกล้ MRT พระราม 9", "สถานีภาวนา")
    * สถานที่สำคัญใกล้เคียง (Landmarks) เช่น ห้างสรรพสินค้า, มหาวิทยาลัย, โรงเรียน
    * ข้อมูลการเดินทาง/ระยะทาง (เช่น "ห่างจาก BTS 200 เมตร")
    * หากไม่มีชื่อโครงการให้รวม: "ที่อยู่เฉพาะเจาะจง > สถานีรถไฟฟ้า > สถานที่ใกล้เคียง"
    * หากมีชื่อโครงการให้ใส่เฉพาะที่อยู่และทำเลสำคัญที่อยู่ใน Post (ไม่ต้องซ้ำกับชื่อโครงการ)
  - ถ้าไม่มีข้อมูลตำแหน่งให้ใส่ "ไม่ระบุ"

รูปแบบคำตอบ:
- ตอบเป็น JSON เดียวบรรทัดเดียวเท่านั้น
- ห้ามใช้ code block
- ห้ามมีข้อความอื่นนอกเหนือ JSON
- ต้องเป็นไปตามสคีมา:
{"extracted": {"property_type": "string", "rental_sale_status": "string", "project_name": "string", "location_text": "string", "price_text": "string", "price_value_thb": float|null}}"""

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\s+", " ", s.replace("\u200b", " ")).strip()
    s = s.replace("ดูน้อยลง", "")
    return s


def clamp_text(s: str, n: int = 3500) -> str:
    if not isinstance(s, str):
        return ""
    return s[:n]


def build_filter_message(row):
    """Build message for real estate filter check"""
    desc = normalize_text(row.get("Description", ""))
    msg = clamp_text(desc, 2000)  # Use less text for filter
    return msg


def build_user_message(row):
    msg = (
        "URL: " + clamp_text(row.get("URL", ""))
        + "\nTITLE: " + clamp_text(row.get("Title", ""))
        + "\nPRICE: " + clamp_text(row.get("Price", ""))
        + "\nDETAILS: " + clamp_text(row.get("Property_Details", ""))
        + "\nDESCRIPTION:\n" + clamp_text(row.get("Description", ""))
    )
    return msg


def extract_json_blob(s: str) -> str:
    s = s.strip().replace("\u200b", "")
    s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s, flags=re.IGNORECASE).strip()
    i = s.find("{")
    j = s.rfind("}")
    if i != -1 and j != -1 and j >= i:
        return s[i : j + 1]
    return s


def parse_extracted(s: str) -> dict:
    d = {}
    jb = extract_json_blob(s)
    m = re.search(r'"?extracted"?\s*:\s*\{(.*)\}', jb, re.S)
    block = m.group(1) if m else ""

    def grab_str(k):
        m2 = re.search(rf'"?{k}"?\s*:\s*"([^"]*)"', block)
        if m2:
            return m2.group(1)
        m3 = re.search(rf"'{k}'\s*:\s*'([^']*)'", block)
        return m3.group(1) if m3 else None

    def grab_float_or_null(k):
        m2 = re.search(
            rf'"?{k}"?\s*:\s*(null|None|[0-9]+(?:\.[0-9]+)?)', block, re.I
        )
        if not m2:
            return None
        v = m2.group(1)
        if v.lower() in ("null", "none"):
            return None
        return float(v)

    d["property_type"] = grab_str("property_type")
    d["rental_sale_status"] = grab_str("rental_sale_status")
    d["project_name"] = grab_str("project_name")
    d["location_text"] = grab_str("location_text")
    d["price_text"] = grab_str("price_text")
    d["price_value_thb"] = grab_float_or_null("price_value_thb")
    return d


def call_typhoon_extract(json_input_text: str, system_prompt: str, max_retries: int = 3) -> str:
    """Call Typhoon API with retry logic for timeout/connection errors"""
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model="typhoon-v2.5-30b-a3b-instruct",
                temperature=0.1,
                max_tokens=16000,
                top_p=0.96,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json_input_text},
                ],
                timeout=60.0,
            )
            return r.choices[0].message.content
        except (APITimeoutError, APIConnectionError) as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                print(f"API timeout/connection error (attempt {attempt + 1}/{max_retries}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"API error after {max_retries} attempts: {e}")
                raise


def is_real_estate_post(row) -> bool:
    """Use AI to filter and check if post is real estate related"""
    try:
        msg = build_filter_message(row)
        raw = call_typhoon_extract(msg, FILTER_PROMPT, max_retries=2)
        blob = extract_json_blob(raw)
        
        try:
            result = json.loads(blob)
            return result.get("is_real_estate", False)
        except:
            # If JSON parsing fails, assume it's real estate to be safe
            return True
    except Exception as e:
        print(f"Filter check error: {e}")
        # If filter fails, skip to be safe
        return False


def format_price_thb(n):
    if n is None:
        return "-"
    if n >= 1_000_000:
        v = round(n / 1_000_000, 2)
        s = f"{v}".rstrip("0").rstrip(".")
        return f"{s} ล้านบาท"
    if n >= 1_000:
        v = round(n / 1_000, 0)
        return f"{int(v)} พันบาท"
    return f"{n} บาท"


def build_row(row, extracted: dict) -> dict:
    ptxt = extracted.get("price_text")
    pval = extracted.get("price_value_thb")
    if isinstance(pval, str):
        cleaned = pval.replace(",", "").strip()
        if cleaned == "" or cleaned.lower() in ("nan", "null", "none"):
            pval = None
        elif re.fullmatch(r"-?\d+(\.\d+)?", cleaned):
            pval = float(cleaned)
        else:
            pval = None
    price_txt = (
        ptxt
        if (isinstance(ptxt, str) and ptxt.strip() != "")
        else (format_price_thb(pval) if pval is not None else "-")
    )
    
    ptype = extracted.get("property_type") or "-"
    status = extracted.get("rental_sale_status") or "-"
    project = extracted.get("project_name") or "-"
    location = extracted.get("location_text") or "-"
    url = row.get("URL", "-")
    
    # Combine project name and location if both exist
    if project and project != "-" and project != "ไม่ระบุ":
        if location and location != "-":
            location_field = f"{project} > {location}"
        else:
            location_field = project
    else:
        location_field = location
    
    # Format: ประเภท, สถานะ, ราคา, ชื่อโครงการ/ทำเล, Link
    return {
        "ประเภท": ptype,
        "สถานะ": status,
        "ราคา": price_txt,
        "ชื่อโครงการ/ทำเล": location_field,
        "Link": url,
    }


def process_and_save(input_path: str, output_path: str):
    print("load_csv:", input_path)
    df = pd.read_csv(input_path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    
    # Map columns to standard names
    if "Post_URL" in df.columns:
        df["URL"] = df["Post_URL"]
    if "Full_Content" in df.columns:
        df["Description"] = df["Full_Content"]
    
    # Default empty for missing columns
    for c in ["Title", "Property_Details", "Price", "Description"]:
        if c not in df.columns:
            df[c] = ""
            
    rows = []
    skipped_rows = []
    filtered_rows = []
    print("start_iterate")
    
    for i, row in df.iterrows():
        try:
            desc = normalize_text(row.get("Description", ""))
            
            # Step 1: Filter - Check if it's real estate
            print(f"filter_check_index: {i}")
            if not is_real_estate_post(row):
                print(f"FILTERED_index: {i} - Not real estate")
                filtered_rows.append(i)
                continue
            
            # Step 2: Extract data from real estate posts
            user_msg = build_user_message({
                "URL": row.get("URL", ""),
                "Title": desc,
                "Price": "",
                "Property_Details": desc,
                "Description": desc,
            })
            
            print(f"typhoon_call_index: {i}")
            raw = call_typhoon_extract(user_msg, SYSTEM_PROMPT)
            blob = extract_json_blob(raw)
            
            # Extract & Build Row
            extracted = parse_extracted(blob)
            out = build_row(row, extracted)
            
            rows.append(out)
            print(f"processed_index: {i}, type: {out['ประเภท']}, status: {out['สถานะ']}, location: {out['ชื่อโครงการ/ทำเล'][:40]}")
            
        except (APITimeoutError, APIConnectionError) as e:
            print(f"SKIPPED_index: {i} - API error: {type(e).__name__}")
            skipped_rows.append({"index": i, "reason": "API_error", "error": str(e)})
        except Exception as e:
            print(f"SKIPPED_index: {i} - Unexpected error: {type(e).__name__}: {e}")
            skipped_rows.append({"index": i, "reason": "unexpected_error", "error": str(e)})

    print(f"\n{'='*50}")
    print(f"Processing Summary:")
    print(f"  Total posts: {len(df)}")
    print(f"  Filtered (not real estate): {len(filtered_rows)}")
    print(f"  Successfully processed: {len(rows)}")
    print(f"  Skipped (errors): {len(skipped_rows)}")
    print(f"{'='*50}")
    
    if skipped_rows:
        print(f"Skipped rows: {[r['index'] for r in skipped_rows]}")
    
    out_df = pd.DataFrame(
        rows, columns=["ประเภท", "สถานะ", "ราคา", "ชื่อโครงการ/ทำเล", "Link"]
    )
    
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    out_df.to_excel(output_path, index=False)
    print("saved_excel:", output_path)


if __name__ == "__main__":
    input_path = r"/home/kongla/Documents/GitHub/Real-estate-Scraping/facebook/facebook_details.csv"
    output_path = r"facebook_output.xlsx"
    process_and_save(input_path, output_path)


load_csv: facebook_details.csv
shape: (501, 3)
columns: ['Post_URL', 'Full_Content', 'Date']
start_iterate
filter_check_index: 0
FILTERED_index: 0 - Not real estate
filter_check_index: 1
typhoon_call_index: 1
processed_index: 1, type: คอนโด, status: ขาย, location: The Address Pathumwan > ติด BTS ราชดำริ,
filter_check_index: 2
typhoon_call_index: 2
processed_index: 2, type: คอนโด, status: เช่า, location: Life Asoke Hype > MRT พระราม9 (300 m.)
filter_check_index: 3
typhoon_call_index: 3
processed_index: 3, type: คอนโด, status: เช่า, location: One9Five Asoke-Rama 9 > MRT พระราม 9 (40
filter_check_index: 4
typhoon_call_index: 4
processed_index: 4, type: คอนโด, status: เช่า, location: The Selected Kaset- Ngamwongwan > BTS Ka
filter_check_index: 5
typhoon_call_index: 5
processed_index: 5, type: คอนโด, status: ขาย, location: The Fine by Fine Home อารีย์ 4 > ใจกลางอ
filter_check_index: 6
typhoon_call_index: 6
processed_index: 6, type: คอนโด, status: เช่า, location: Serio Sukhumvit 50 > ซอยสุขุ

In [1]:
import json
import os
import re
import time
import csv
import logging
from datetime import datetime, timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

INPUT_PATH = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping/facebook/facebook_details.csv")
OUTPUT_PATH = Path("/home/kongla/Documents/GitHub/Real-estate-Scraping/facebook/facebook_output.csv")

MODEL_NAME = "typhoon-v2.5-30b-a3b-instruct"
MAX_WORKERS = 5
LLM_TIMEOUT = 60.0

client = OpenAI(
    api_key=os.getenv("TYPHOON_API_KEY"),
    base_url="https://api.opentyphoon.ai/v1",
    timeout=LLM_TIMEOUT,
)

SYSTEM_PROMPT = """คุณคือ AI วิเคราะห์อสังหาริมทรัพย์
ดึงข้อมูลจากข้อความที่ให้มา และตอบกลับเป็น JSON เท่านั้น
{
"is_real_estate": true/false,
"post_date_text": "ดึงข้อความวันที่ที่พบในข้อมูล",
"extracted":{
"property_type":"",
"rental_sale_status":"",
"project_name":"",
"district":"",
"size_text":"",
"price_text":"",
"price_value_thb":null,
"phone":"",
"line":"",
"description":""
}
}
"""

MONTH_MAP = {
    "มกราคม": 1, "ม.ค.": 1, "กุมภาพันธ์": 2, "ก.พ.": 2, "มีนาคม": 3, "มี.ค.": 3,
    "เมษายน": 4, "เม.ย.": 4, "พฤษภาคม": 5, "พ.ค.": 5, "มิถุนายน": 6, "มิ.ย.": 6,
    "กรกฎาคม": 7, "ก.ค.": 7, "สิงหาคม": 8, "ส.ค.": 8, "กันยายน": 9, "ก.ย.": 9,
    "ตุลาคม": 10, "ต.ค.": 10, "พฤศจิกายน": 11, "พ.ย.": 11, "ธันวาคม": 12, "ธ.ค.": 12,
}

OUTPUT_HEADERS = [
    "วันที่โพส", "website", "ประเภท", "สถานะ", "ชื่อโครงการ", 
    "ขนาด", "ราคา", "เขต", "Link", "เบอร์โทรศัพท์", "Line", "คำอธิบาย"
]

def normalize_json_string(text: str) -> str:
    cleaned = re.sub(r"^```.*?\n|\n```$", "", text.strip(), flags=re.S)
    start, end = cleaned.find("{"), cleaned.rfind("}")
    return cleaned[start : end + 1] if -1 < start < end else ""

def call_llm_service(payload: str) -> dict | None:
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                temperature=0.1,
                max_tokens=16000,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": payload},
                ],
                response_format={"type": "json_object"}
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            logger.warning(f"Retry {attempt + 1}: {str(e)}")
            time.sleep(2 ** attempt)
    return None

def parse_date(date_text: str) -> str:
    now = datetime.now()
    if not date_text or date_text == "-": return "-"
    
    val = date_text.strip().lower()
    if any(k in val for k in ("วันนี้", "นาที", "ชั่วโมง", "ชม.")):
        return now.strftime("%d/%m/%Y")
    
    if "เมื่อวาน" in val:
        return (now - timedelta(days=1)).strftime("%d/%m/%Y")
        
    m_days = re.search(r"(\d+)\s*วัน", val)
    if m_days:
        return (now - timedelta(days=int(m_days.group(1)))).strftime("%d/%m/%Y")

    m_date = re.search(r"(\d{1,2})[\s\.\-/]+([ก-๙a-zA-Z]+)(?:[\s\.\-/]+(\d{2,4}))?", val)
    if m_date:
        d, m_raw = int(m_date.group(1)), m_date.group(2)
        m = MONTH_MAP.get(m_raw)
        if m:
            y = now.year
            if m_date.group(3):
                y_raw = int(m_date.group(3))
                y = y_raw - 543 if y_raw > 2400 else (y_raw if y_raw > 100 else 2000 + y_raw)
            return f"{d:02d}/{m:02d}/{y}"
            
    return "-"

def transform_record(raw_row: dict, ai_data: dict) -> dict:
    ext = ai_data.get("extracted", {})
    return {
        "วันที่โพส": parse_date(ai_data.get("post_date_text", "")),
        "website": "facebook",
        "ประเภท": ext.get("property_type", "-"),
        "สถานะ": ext.get("rental_sale_status", "-"),
        "ชื่อโครงการ": ext.get("project_name", "-"),
        "ขนาด": ext.get("size_text", "-"),
        "ราคา": ext.get("price_text", "-"),
        "เขต": ext.get("district", "-"),
        "Link": raw_row.get("Post_URL") or raw_row.get("URL", "-"),
        "เบอร์โทรศัพท์": ext.get("phone", "-"),
        "Line": ext.get("line", "-"),
        "คำอธิบาย": ext.get("description", "-"),
    }

def worker(row: dict) -> dict | None:
    context = "\n".join([f"{k}: {v}" for k, v in row.items() if v])[:2000]
    ai_response = call_llm_service(context)
    
    if ai_response and ai_response.get("is_real_estate"):
        return transform_record(row, ai_response)
    return None

def main():
    if not OUTPUT_PATH.exists():
        with open(OUTPUT_PATH, 'w', encoding='utf-8-sig', newline='') as f:
            csv.DictWriter(f, fieldnames=OUTPUT_HEADERS).writeheader()

    with open(INPUT_PATH, 'r', encoding='utf-8-sig') as f_in:
        reader = csv.DictReader(f_in)
        
        with open(OUTPUT_PATH, 'a', encoding='utf-8-sig', newline='') as f_out:
            writer = csv.DictWriter(f_out, fieldnames=OUTPUT_HEADERS)
            
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = {executor.submit(worker, row): row for row in reader}
                
                for count, future in enumerate(futures, 1):
                    try:
                        result = future.result()
                        if result:
                            writer.writerow(result)
                            f_out.flush()
                        
                        if count % 10 == 0:
                            logger.info(f"Processed {count} rows...")
                            
                    except Exception as e:
                        logger.error(f"Error processing row: {str(e)}")

if __name__ == "__main__":
    main()

2026-04-13 13:50:30,894 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:33,752 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:36,977 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:44,108 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:44,575 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:44,686 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:47,857 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:47,858 - INFO - HTTP Request: POST https://api.opentyphoon.ai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-13 13:50:59,281 - INFO - HTTP Request: POST https://api.